# Memory and Chat History

Chat models are **stateless** — each API call only sees the messages you send. **Memory** is how your application carries **conversation history** across turns: prior questions, answers, tool results, and summaries. LangChain 1.x does not use legacy `ConversationBufferMemory`; instead you manage **`list[BaseMessage]`** explicitly, wrap chains with **`RunnableWithMessageHistory`**, checkpoint **agents** with **`checkpointer` + `thread_id`** (**13**), and trim or summarize when contexts grow too large.

This notebook covers manual history lists, **`MessagesPlaceholder`** (**04**), **`RunnableWithMessageHistory`**, session stores, **`trim_messages`**, agent checkpointing, **`get_state`**, multi-thread isolation, **`SummarizationMiddleware`**, conversational RAG memory hooks (**11**), and production session patterns.

Prerequisites: **03. Chat Models and Messages**, **04. Prompt Templates**, **11. RAG with LCEL**, **13. Agents with create_agent**.

In [ ]:
OPENAI_API_KEY = "sk-proj-placeholder-replace-with-your-real-key"

import json

import langchain_core

print("langchain-core:", langchain_core.__version__)

---

## 1. Why Memory Matters

Without memory, every user message is a **first message**:

```
Turn 1: "Tell me about Alembic migrations."
Turn 2: "What command do I run?"     ← meaningless without Turn 1
```

| Layer | What it stores | LangChain mechanism |
|-------|----------------|---------------------|
| **Prompt context** | Messages sent to the model this call | `list[BaseMessage]` |
| **Session store** | Messages across HTTP requests | `InMemoryChatMessageHistory`, DB |
| **Agent graph state** | Full agent trace including tools | `checkpointer` + `thread_id` |
| **Long-context control** | Trimmed or summarized history | `trim_messages`, `SummarizationMiddleware` |

Memory is **not magic** — it is **message list management** with optional persistence.

---

## 2. Setup

Shared model and a tiny documentation knowledge base:

In [ ]:
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini", api_key=OPENAI_API_KEY, temperature=0)

DOCS = {
    "alembic": "Alembic applies SQLAlchemy migrations. Run alembic upgrade head.",
    "jwt": "JWT bearer tokens use Authorization: Bearer header.",
    "pytest": "Pytest DB fixtures belong in conftest.py.",
}

---

## 3. Manual History — The Foundation

The simplest pattern from **03. Chat Models and Messages**: append each turn to a Python list.

In [ ]:
history = [SystemMessage(content="You help with API documentation. Be brief.")]


def chat_manual(user_text: str) -> str:
    history.append(HumanMessage(content=user_text))
    ai = llm.invoke(history)
    history.append(ai)
    return ai.content


print("Turn 1:", chat_manual("What is Alembic for?"))
print("Turn 2:", chat_manual("What command upgrades the schema?"))
print("messages in history:", len(history))

**Pros:** transparent, easy to debug. **Cons:** you must persist `history` yourself, trim when long, and wire it into every route. LangChain wrappers automate the read/write cycle.

---

## 4. MessagesPlaceholder in LCEL Chains

**`MessagesPlaceholder("history")`** injects a message list into **`ChatPromptTemplate`** (**04**):

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

chat_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant for backend engineers."),
    MessagesPlaceholder("history"),
    ("human", "{input}"),
])

chat_chain = chat_prompt | llm | StrOutputParser()

session_history = [
    HumanMessage(content="What is Alembic?"),
    AIMessage(content="Alembic manages SQLAlchemy database schema migrations."),
]

answer = chat_chain.invoke({
    "history": session_history,
    "input": "What command applies migrations?",
})
print(answer)

The chain does **not** update `session_history` automatically — something must append the new human + AI messages after each call (**§5**).

---

## 5. RunnableWithMessageHistory

**`RunnableWithMessageHistory`** wraps a chain and **reads/writes** a per-session message store:

In [ ]:
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

SESSION_STORE: dict[str, InMemoryChatMessageHistory] = {}


def get_session_history(session_id: str) -> InMemoryChatMessageHistory:
    if session_id not in SESSION_STORE:
        SESSION_STORE[session_id] = InMemoryChatMessageHistory()
    return SESSION_STORE[session_id]


chain_with_history = RunnableWithMessageHistory(
    chat_chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="history",
)

session_config = {"configurable": {"session_id": "user-alice"}}

print(chain_with_history.invoke({"input": "What is JWT used for?"}, config=session_config))
print(chain_with_history.invoke({"input": "What header carries it?"}, config=session_config))

### 5.1 What Happens Under the Hood

```
invoke({input}, session_id)
    │
    ├─ load history from get_session_history(session_id)
    ├─ build prompt: system + history + new human(input)
    ├─ run chain → AI answer
    └─ append HumanMessage(input) + AIMessage(answer) to store
```

| Parameter | Role |
|-----------|------|
| **`get_session_history`** | Factory: `session_id → BaseChatMessageHistory` |
| **`input_messages_key`** | Which input field is the new user message |
| **`history_messages_key`** | Placeholder name in the prompt (`"history"`) |
| **`configurable.session_id`** | Selects the session in `config` |

### 5.2 Session Isolation

Different **`session_id`** values get independent histories:

In [ ]:
alice_cfg = {"configurable": {"session_id": "alice"}}
bob_cfg = {"configurable": {"session_id": "bob"}}

chain_with_history.invoke({"input": "My name is Alice."}, config=alice_cfg)
chain_with_history.invoke({"input": "My name is Bob."}, config=bob_cfg)

print("alice:", chain_with_history.invoke({"input": "What is my name?"}, config=alice_cfg))
print("bob:  ", chain_with_history.invoke({"input": "What is my name?"}, config=bob_cfg))

In production, map **`session_id`** to authenticated user + conversation id. Back **`InMemoryChatMessageHistory`** with Redis, Postgres, or a hosted store for durability.

---

## 6. Inspecting Stored History

Debug sessions by reading the underlying store:

In [ ]:
alice_history = get_session_history("alice")
print(f"alice has {len(alice_history.messages)} messages:")
for msg in alice_history.messages:
    preview = str(msg.content)[:60].replace("\n", " ")
    print(f"  [{msg.type}] {preview}...")

---

## 7. trim_messages — Context Window Limits

Long histories exceed model **context windows** and increase cost. **`trim_messages`** keeps the most recent messages within a token budget:

In [ ]:
from langchain_core.messages import trim_messages

long_history = [SystemMessage(content="You are helpful.")]
for i in range(12):
    long_history.append(HumanMessage(content=f"Question {i}: explain topic {i} briefly."))
    long_history.append(AIMessage(content=f"Answer {i}: topic {i} is about backend engineering concept {i}."))

print("full history messages:", len(long_history))

trimmed = trim_messages(
    long_history,
    max_tokens=200,
    token_counter=llm,
    strategy="last",
    include_system=True,
)

print("trimmed messages:", len(trimmed))
print("first trimmed:", trimmed[0].type, "| last trimmed:", trimmed[-1].type)

| Parameter | Meaning |
|-----------|--------|
| **`max_tokens`** | Budget for retained messages |
| **`strategy="last"`** | Keep most recent messages |
| **`strategy="first"`** | Keep earliest (rare) |
| **`include_system=True`** | Always retain system message |
| **`token_counter`** | `llm`, custom callable, or `"approximate"` |

Apply trimming **before** `llm.invoke` or inside middleware — not after the user has already sent an oversized payload.

### 7.1 Trim Inside a Chain

Wrap trimming as a **`RunnableLambda`** before the model:

In [ ]:
from langchain_core.runnables import RunnableLambda


def trim_history_dict(inputs: dict) -> dict:
    trimmed = trim_messages(
        inputs["history"],
        max_tokens=150,
        token_counter="approximate",
        strategy="last",
        include_system=True,
    )
    return {**inputs, "history": trimmed}


trimmed_chat_chain = (
    RunnableLambda(trim_history_dict)
    | chat_prompt
    | llm
    | StrOutputParser()
)

print(trimmed_chat_chain.invoke({
    "history": long_history,
    "input": "Summarize our conversation in one sentence.",
}))

---

## 8. Agent Memory — checkpointer and thread_id

**13. Agents with create_agent** previewed checkpointing. Agents store the **full message trace** (including tool calls) per **`thread_id`**:

In [ ]:
from langchain.agents import create_agent
from langchain_core.tools import tool
from langgraph.checkpoint.memory import MemorySaver


@tool
def lookup_doc(topic: str) -> str:
    """Look up documentation for alembic, jwt, or pytest."""
    key = topic.lower().strip()
    for name, text in DOCS.items():
        if name in key or key in name:
            return text
    return "No doc found for that topic."


def last_ai_content(result: dict) -> str:
    for msg in reversed(result["messages"]):
        if isinstance(msg, AIMessage) and msg.content:
            return msg.content if isinstance(msg.content, str) else str(msg.content)
    return ""


checkpointer = MemorySaver()
agent = create_agent(
    model=llm,
    tools=[lookup_doc],
    system_prompt="Use lookup_doc for factual questions about the codebase.",
    checkpointer=checkpointer,
)

thread_cfg = {"configurable": {"thread_id": "support-ticket-42"}}

r1 = agent.invoke(
    {"messages": [HumanMessage(content="Tell me about Alembic migrations.")]},
    config=thread_cfg,
)
print("turn 1:", last_ai_content(r1)[:100], "...")

r2 = agent.invoke(
    {"messages": [HumanMessage(content="What command do I run?")]},
    config=thread_cfg,
)
print("turn 2:", last_ai_content(r2))

Each invoke only passes the **new** `HumanMessage` — the checkpointer merges prior state for the thread.

---

## 9. Reading Agent State with get_state

Inspect or resume from stored graph state:

In [ ]:
snapshot = agent.get_state(thread_cfg)
stored_messages = snapshot.values["messages"]

print(f"checkpoint has {len(stored_messages)} messages")
for msg in stored_messages[-4:]:
    preview = str(msg.content)[:55].replace("\n", " ")
    print(f"  [{msg.type}] {preview}...")

**`get_state_history`** returns prior checkpoints for time-travel debugging. Production checkpointers (Postgres, Redis) persist across process restarts — **`MemorySaver`** is for notebooks and tests only.

---

## 10. SummarizationMiddleware — Compress Old Turns

Trimming drops messages; **summarization** compresses them into a compact system note. **`SummarizationMiddleware`** triggers when history exceeds a threshold:

In [ ]:
from langchain.agents.middleware import SummarizationMiddleware

summary_agent = create_agent(
    model=llm,
    tools=[lookup_doc],
    system_prompt="You are a patient support agent for backend docs.",
    checkpointer=MemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=llm,
            trigger=("messages", 10),
            keep=("messages", 4),
        ),
    ],
)

summary_thread = {"configurable": {"thread_id": "long-thread-1"}}

topics = ["alembic", "jwt", "pytest", "alembic again", "jwt header", "pytest fixtures"]
for topic in topics:
    summary_agent.invoke(
        {"messages": [HumanMessage(content=f"Tell me about {topic}.")]},
        config=summary_thread,
    )

final_state = summary_agent.get_state(summary_thread)
print("messages after summarization middleware:", len(final_state.values["messages"]))

| Parameter | Meaning |
|-----------|--------|
| **`trigger`** | When to summarize — e.g. `("messages", 10)` |
| **`keep`** | Recent messages to retain verbatim |
| **`model`** | LLM that writes the summary |

Use summarization when **losing old detail** is acceptable; use trimming when you only need recent turns.

---

## 11. Conversational RAG + Memory

**11. RAG with LCEL** condensed follow-up questions before retrieval. Combine that with session history:

In [ ]:
from langchain_core.prompts import MessagesPlaceholder

condense_prompt = ChatPromptTemplate.from_messages([
    MessagesPlaceholder("history"),
    (
        "human",
        "Rewrite this follow-up as a standalone question:\n{question}",
    ),
])

condense_chain = condense_prompt | llm | StrOutputParser()

rag_history = [
    HumanMessage(content="Tell me about Alembic migrations."),
    AIMessage(content="Alembic applies SQLAlchemy schema migrations."),
]

standalone_q = condense_chain.invoke({
    "history": rag_history,
    "question": "What command do I run?",
})
print("standalone question:", standalone_q)
print("doc lookup:", DOCS["alembic"])

**Pattern:** session store holds **`history`** → condense step produces **`standalone_question`** → retriever/RAG uses standalone → append new human + AI to store. Agents with a search tool approximate this by choosing when to retrieve.

---

## 12. What Belongs in Memory?

| Include | Exclude (usually) |
|---------|-------------------|
| User questions | Raw retrieved doc dumps (re-fetch instead) |
| Final assistant answers | Duplicate system prompts per turn |
| Key tool **results** the user needs | Full intermediate tool JSON blobs |
| Summaries of old turns | Massive error stack traces |

Agent checkpointers store **tool traces** by default — good for debugging, costly for tokens. Use **`SummarizationMiddleware`** or custom middleware to prune tool noise in long sessions.

---

## 13. FastAPI Session Pattern (Sketch)

Typical web app wiring — not framework-specific, but shows where memory sits:

In [ ]:
def handle_chat_request(user_id: str, conversation_id: str, user_text: str) -> str:
    """Pseudocode for a chat endpoint."""
    session_id = f"{user_id}:{conversation_id}"
    config = {"configurable": {"session_id": session_id}}
    return chain_with_history.invoke({"input": user_text}, config=config)


def handle_agent_request(thread_id: str, user_text: str) -> str:
    """Agent endpoint with checkpointed memory."""
    config = {"configurable": {"thread_id": thread_id}}
    result = agent.invoke(
        {"messages": [HumanMessage(content=user_text)]},
        config=config,
    )
    return last_ai_content(result)


print(handle_chat_request("u1", "c9", "What is JWT?")[:80], "...")
print(handle_agent_request("support-ticket-42", "And the command again?")[:80], "...")

Use **`ainvoke`** in async FastAPI routes (**07. Streaming, Batch, and Async**). Observability hooks are in **15. Callbacks, Caching, and Observability**.

---

## 14. Memory Strategy Guide

| Scenario | Recommended approach |
|----------|---------------------|
| Simple multi-turn chat chain | **`RunnableWithMessageHistory`** |
| Tool-using assistant | **`create_agent` + `checkpointer` + `thread_id`** |
| Long support conversations | **`SummarizationMiddleware`** or periodic trim |
| Strict token budget | **`trim_messages`** before each call |
| Stateless API (no memory) | Pass full history from client each request |
| Multi-user SaaS | Durable store + per-user `session_id` / `thread_id` |
| RAG follow-ups | Condense question + session history (**11**) |

---

## 15. Common Mistakes

| Mistake | Consequence | Fix |
|---------|-------------|-----|
| Not appending AI replies | Model forgets its own answers | Use `RunnableWithMessageHistory` or manual append |
| Same `session_id` for all users | Cross-user data leak | Unique id per user + conversation |
| Unbounded history | Context overflow / cost | `trim_messages` or summarization |
| Expecting chain to remember alone | History never grows | Wire session store |
| Agent without checkpointer | Follow-ups fail | Pass `checkpointer` |
| Passing full history + new message duplicated | Repeated turns | Only send **new** message to checkpointed agent |
| Storing huge RAG context in memory | Token bloat | Store answer + citation ids, re-retrieve later |
| `MemorySaver` in production | State lost on restart | Postgres/Redis checkpointer |

---

## 16. Summary

**Memory** in LangChain 1.x is **message list management**: build `list[BaseMessage]`, inject via **`MessagesPlaceholder`**, persist with **`RunnableWithMessageHistory`** or agent **`checkpointer`**, and control size with **`trim_messages`** / **`SummarizationMiddleware`**.

Key takeaways:

- Models are stateless — **you** supply history every call unless a wrapper persists it.
- **`RunnableWithMessageHistory`** automates read/write for LCEL chat chains.
- **`session_id`** isolates chain sessions; **`thread_id`** isolates agent checkpoints.
- **`trim_messages`** enforces token budgets; summarization preserves gist of old turns.
- **Agents** remember tool traces — plan for growth on long threads.
- **Conversational RAG** needs history + condense before retrieval (**11**).

Demonstrations covered manual history, `MessagesPlaceholder`, session-wrapped chains, trimming, checkpointed agents, state inspection, summarization middleware, and conversational RAG memory hooks.

Next: **15. Callbacks, Caching, and Observability** — tracing, logging, and cache layers across chains and agents.